# Proyecto RaSA — Entendimiento de datos
**Entregable 1 — Perfilamiento de datos**

Se exploran las cuatro fuentes compartidas por RaSA en la base `RaSaTransaccional` para entender su estructura, volumen y contenido, y dejar listas las observaciones que alimentan el análisis de calidad (Entregable 2) y las conclusiones de viabilidad (Entregable 3).

Fuentes:
- **F1. FuenteAreasDeServicio_Copia_E** — áreas de servicio por condado.
- **F2. FuenteTiposBeneficio_Copia_E** — tipos de beneficio.
- **F3. FuentePlanesBeneficio_Copia_E** — beneficios ofrecidos por cada plan (tabla central).
- **F4. FuenteCondicionesDePago_Copia_E** — condiciones de pago (referencia).

## 0. Configuración y utilidades

In [ ]:
# Credenciales y conexión (use su usuario/clave del curso y su servidor de los tutoriales)
db_user = ''
db_psswd = ''

# Base de datos de las fuentes
source_db_connection_string = 'jdbc:mysql://157.253.236.120:8080/RaSaTransaccional'  # AJUSTE host/puerto al suyo

# Driver JDBC de MySQL
path_jar_driver = '/workspaces/tareas-modelado-datos-2026-13/drivers/mysql-connector-j.jar' 

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder \
    .appName('Perfilamiento_RaSA') \
    .config('spark.jars', path_jar_driver) \
    .config('spark.driver.extraClassPath', path_jar_driver) \
    .config('spark.executor.extraClassPath', path_jar_driver) \
    .getOrCreate()

spark.conf.set('spark.sql.legacy.timeParserPolicy', 'LEGACY')

In [ ]:
def obtener_dataframe_de_bd(db_connection_string, sql, db_user, db_psswd):
    return spark.read.format('jdbc') \
        .option('url', db_connection_string) \
        .option('dbtable', sql) \
        .option('user', db_user) \
        .option('password', db_psswd) \
        .option('driver', 'com.mysql.cj.jdbc.Driver') \
        .load()

def cargar_tabla(nombre):
    sql = f'(SELECT * FROM RaSaTransaccional.{nombre}) AS t'
    return obtener_dataframe_de_bd(source_db_connection_string, sql, db_user, db_psswd)

In [ ]:
# --- Funciones de perfilamiento reutilizables ---
def resumen(df, nombre):
    print(f'=== {nombre} ===')
    print('Filas:', df.count(), '| Columnas:', len(df.columns))
    df.printSchema()

def nulos(df):
    total = df.count()
    fila = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).collect()[0]
    print(f'Total filas: {total}')
    for c in df.columns:
        n = fila[c]
        print(f'  {c}: {n} nulos ({100*n/total:.1f}%)')

def cardinalidad(df):
    fila = df.select([F.countDistinct(F.col(c)).alias(c) for c in df.columns]).collect()[0]
    for c in df.columns:
        print(f'  {c}: {fila[c]} valores distintos')

def frecuencias(df, columna, n=20):
    df.groupBy(columna).count().orderBy('count', ascending=False).show(n, truncate=False)

def anio(colname='Fecha'):
    # Extrae el año como texto (robusto aunque la fecha venga como string)
    return F.substring(F.col(colname).cast('string'), 1, 4)

Antes de perfilar, confirme los nombres reales de las tablas (en el diccionario una aparece como *FuentesPlanesBeneficio*, con 's'). Ajuste el diccionario `TABLAS` si alguno difiere.

In [ ]:
# Verifique los nombres reales de las fuentes
tablas_bd = obtener_dataframe_de_bd(source_db_connection_string,
    "(SELECT table_name FROM information_schema.tables WHERE table_schema='RaSaTransaccional' AND table_name LIKE 'Fuente%') AS t",
    db_user, db_psswd)
tablas_bd.show(truncate=False)

TABLAS = {
    'areas':        'FuenteAreasDeServicio_Copia_E',
    'tipos':        'FuenteTiposBeneficio_Copia_E',
    'planes':       'FuentePlanesBeneficio_Copia_E',   # ajuste si en la BD es 'FuentesPlanes...'
    'condiciones':  'FuenteCondicionesDePago_Copia_E',
}

## F1 — FuenteAreasDeServicio_Copia_E
**Significado de una fila:** un área de servicio definida sobre un condado (con su población, área y densidad). El negocio indica que deben ser **5409 áreas** y que cubren todos los condados del país.

In [ ]:
areas = cargar_tabla(TABLAS['areas'])
resumen(areas, 'Áreas de servicio')

Conteo de nulos por columna (completitud) y cardinalidad (¿es `idAreaServicio_T` única?).

In [ ]:
print('--- Nulos ---'); nulos(areas)
print('--- Valores distintos ---'); cardinalidad(areas)

Estadísticos de las columnas numéricas y revisión de las cifras del negocio.

In [ ]:
areas.select('PoblacionAct', 'Area', 'Densidad').describe().show()

print('Áreas distintas (esperado 5409):', areas.select('idAreaServicio_T').distinct().count())
print('Condados distintos:', areas.select('idGeografía_T').distinct().count())
print('--- Estados ---'); frecuencias(areas, 'Estado')
print('--- Años (Fecha) ---'); areas.select(anio().alias('anio')).groupBy('anio').count().orderBy('anio').show()

**Observaciones (a comentar):** ¿el número de áreas distintas coincide con 5409? ¿Hay `idAreaServicio_T` repetidos (afecta unicidad)? ¿Hay condados/estados nulos o densidades negativas/cero? ¿El rango de años cae en 2017–2019 como dice el negocio?

## F2 — FuenteTiposBeneficio_Copia_E
**Significado de una fila:** un tipo de beneficio y sus atributos (si es EHB, si tiene límite cuantitativo, exclusiones, etc.). El negocio indica **170 tipos de beneficio**.

In [ ]:
tipos = cargar_tabla(TABLAS['tipos'])
resumen(tipos, 'Tipos de beneficio')

In [ ]:
print('--- Nulos ---'); nulos(tipos)
print('--- Valores distintos ---'); cardinalidad(tipos)
print('Tipos distintos (esperado 170):', tipos.select('idTipoBeneficio_T').distinct().count())

Distribución de las banderas (booleanas) y de la unidad del límite.

In [ ]:
for c in ['EsEHB', 'EstaCubiertoPorSeguro', 'TieneLimiteCuantitativo',
          'ExcluidoDelDesembolsoMaximoDentroDeLaRed', 'ExcluidoDelDesembolsoMaximoFueraDeLaRed']:
    print(f'--- {c} ---'); frecuencias(tipos, c)
print('--- UnidadDelLimite ---'); frecuencias(tipos, 'UnidadDelLimite')

**Observaciones (a comentar):** ¿salen 170 tipos? ¿`idTipoBeneficio_T` es único? ¿Las banderas tienen solo valores válidos (0/1, true/false) o hay valores raros (validez)? ¿Cuántos tienen `TieneLimiteCuantitativo` = sí (se cruzará con `cantidadLimite` en planes)?

## F3 — FuentePlanesBeneficio_Copia_E
**Significado de una fila:** un beneficio concreto ofrecido por un plan, en un área de servicio, con sus valores de copago/coseguro y condiciones. Es la **tabla central** que relaciona todas las demás. El negocio indica **301 planes en 2017 y 422 en 2018**, y para 2018 los máximos de copago/coseguro son **3300 y 100**.

In [ ]:
planes = cargar_tabla(TABLAS['planes'])
resumen(planes, 'Planes-beneficio')

In [ ]:
print('--- Nulos ---'); nulos(planes)
print('--- Valores distintos ---'); cardinalidad(planes)

Planes distintos por año (validación 301 en 2017, 422 en 2018) y estadísticos de los valores.

In [ ]:
planes.withColumn('anio', anio()).groupBy('anio') \
      .agg(F.countDistinct('idPlan_T').alias('planes_distintos')) \
      .orderBy('anio').show()

planes.select('valorCopago', 'valorCoseguro', 'cantidadLimite').describe().show()

print('--- Máximos copago/coseguro 2018 (esperado 3300 y 100) ---')
planes.withColumn('anio', anio()).filter(F.col('anio') == '2018') \
      .select(F.max('valorCopago').alias('max_copago'),
              F.max('valorCoseguro').alias('max_coseguro')).show()

**Observaciones (a comentar):** ¿los planes por año coinciden con 301/422? ¿Los máximos 2018 coinciden con 3300/100 (validez)? ¿Hay `valorCopago`/`valorCoseguro` negativos o nulos? ¿Las FK (`idTipoBeneficio_T`, `idAreaDeServicio_T`, `idCondicionesDePago...`) tienen nulos o valores que no existen en las otras fuentes (consistencia/integridad referencial)? ¿Los beneficios con límite cuantitativo tienen `cantidadLimite` ≠ 0?

## F4 — FuenteCondicionesDePago_Copia_E
**Significado de una fila:** una condición de pago (referencia), de tipo Copago, Coseguro o Anticipado. El negocio indica **15 condiciones de copago y 5 de coseguro**.

In [ ]:
condiciones = cargar_tabla(TABLAS['condiciones'])
resumen(condiciones, 'Condiciones de pago')

In [ ]:
print('--- Nulos ---'); nulos(condiciones)
print('--- Valores distintos ---'); cardinalidad(condiciones)
print('--- Tipo (esperado: 15 Copago, 5 Coseguro) ---'); frecuencias(condiciones, 'Tipo')
print('--- Descripcion ---'); frecuencias(condiciones, 'Descripcion')

**Observaciones (a comentar):** ¿hay 15 de Copago y 5 de Coseguro? ¿`idCondicionesDePago_T` es único? ¿`Tipo` solo toma los valores válidos (Copago/Coseguro/Anticipado)?

## Relaciones entre fuentes (insumo para el Entregable 3)
La tabla central `FuentePlanesBeneficio_Copia_E` integra las demás mediante estas llaves:

| Desde (Planes) | Hacia | Columna destino |
|---|---|---|
| idTipoBeneficio_T | FuenteTiposBeneficio_Copia_E | idTipoBeneficio_T |
| idAreaDeServicio_T | FuenteAreasDeServicio_Copia_E | idAreaServicio_T |
| idCondicionesDePagoCopago_T | FuenteCondicionesDePago_Copia_E | idCondicionesDePago_T (rol Copago) |
| idCondicionesDePagoCoseguro_T | FuenteCondicionesDePago_Copia_E | idCondicionesDePago_T (rol Coseguro) |

Además, `FuenteAreasDeServicio_Copia_E.idGeografía_T` identifica el condado (geografía). Estas relaciones se validan en el Entregable 3 (integridad referencial) y sustentan los análisis de cobertura, concentración y comportamiento de proveedores.